# Day K1 — 멀티모달 FOG 데이터셋 해부: 파일 구조, 라벨, 그리고 "직접 세어 보기"

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/K-multimodal-fog/day-K1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/K-multimodal-fog/day-K1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 합성 라벨로 run-length 연습: 사건 수, 총 시간, 해상도

In [ ]:
import numpy as np
import pandas as pd

# ---- 합성 데이터 뼈대 (K1–K5 공통) ----
fs = 500                                   # 500행 = 1초
t = np.arange(0, 60, 1/fs)                 # 60초
label = ((t >= 25) & (t < 35)).astype(int) # 25–35초가 FOG(가정)
label[(t >= 48) & (t < 50.5)] = 1          # 오늘만 덧붙임: 짧은 FOG 하나 더(48–50.5초, 가정)
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)

# ---- 인덱스 → 초 ----
idx = np.arange(t.size)                    # 실제 파일의 0번 열처럼 0, 1, 2, ...
print("행 12500은", idx[12500] / fs, "초")

# ---- run-length: 0→1이 되는 곳이 시작, 1→0이 되는 곳이 끝 ----
def fog_events(label, fs):
    edges = np.diff(np.r_[0, label, 0])    # 앞뒤에 0을 붙여 맨 처음·맨 끝 사건도 잡는다
    starts = np.where(edges == 1)[0]       # 시작 행(포함)
    ends = np.where(edges == -1)[0]        # 끝 행(포함하지 않음)
    return pd.DataFrame({"start_s": starts / fs, "end_s": ends / fs,
                         "dur_s": (ends - starts) / fs, "n_rows": ends - starts})

ev = fog_events(label, fs)
print(ev.to_string(index=False))
print("사건 수:", len(ev), "| 총 FOG 시간: %.1f초" % ev["dur_s"].sum(),
      "| 기록 중 FOG 비율: %.1f%%" % (100 * label.mean()))
print("사건 길이를 1초(500행)로 나눈 나머지:", (ev["n_rows"] % fs).tolist())

### 예시 2 — 여러 모달을 한 시간축에: `subplots(sharex=True)`

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- 합성 데이터 뼈대 ----
fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)
# ---- 오늘 덧붙임: 전경골근 EMG (설명용 가정) ----
# 걷기: 걸음마다 짧은 burst(1 Hz 리듬) / FOG: 리듬 없이 중간 세기로 계속 켜져 있다고 가정
burst = (np.sin(2*np.pi*1.0*t) > 0.5).astype(float)
amp = np.where(label == 0, 0.1 + 0.9*burst, 0.5)
emg = amp * rng.standard_normal(t.size)

onset = np.flatnonzero(np.diff(label) == 1)[0] + 1      # 첫 FOG 시작 행
seg = slice(max(onset - 30*fs, 0), min(onset + 30*fs, t.size))
tt = t[seg] - t[onset]                                   # FOG 시작을 0초로

fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True,
                       gridspec_kw={"height_ratios": [2, 2, 1]})
ax[0].plot(tt, acc[seg], lw=0.6); ax[0].set_ylabel("shank ACC (a.u.)")
ax[1].plot(tt, emg[seg], lw=0.3, color="tab:green"); ax[1].set_ylabel("TA EMG (a.u.)")
ax[2].step(tt, label[seg], where="post", color="tab:red"); ax[2].set_ylabel("label")
ax[2].set_yticks([0, 1]); ax[2].set_xlabel("time from FOG onset (s)")
for a in ax:
    a.axvline(0, color="k", ls="--", lw=0.8)
fig.suptitle("Synthetic: ACC + EMG + label around FOG onset (assumed shapes)")
fig.tight_layout()
os.makedirs("figures/advanced/K", exist_ok=True)
fig.savefig("figures/advanced/K/K1_synthetic_multiaxis.png", dpi=120)
print("FOG 시작: %.1f초 (행 %d) | 그림 구간: %.1f ~ %.1f초" % (t[onset], onset, tt[0], tt[-1]))
pre, post = slice(onset - 10*fs, onset), slice(onset, onset + 10*fs)
print("시작 전 10초 |EMG| 평균 %.2f, 시작 후 10초 %.2f" % (np.abs(emg[pre]).mean(), np.abs(emg[post]).mean()))
print("저장: figures/advanced/K/K1_synthetic_multiaxis.png")

## 직접 해보기

**1. 인덱스 → 초.** 행 17250은 과제 시작 후 몇 초입니까?

In [ ]:
import numpy as np
fs = 500
t = np.arange(0, 60, 1/fs)
idx = np.arange(t.size)
print(idx[17250] / ____)

In [ ]:
print(idx[17250] / fs)

**2. 사건의 시작과 끝 찾기.** 예시 1의 라벨(FOG 두 개)에서 시작과 끝을 초로 출력하세요.

In [ ]:
import numpy as np
fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
label[(t >= 48) & (t < 50.5)] = 1
edges = np.diff(np.r_[0, label, 0])
starts = np.flatnonzero(edges == ____)
ends = np.flatnonzero(edges == ____)
print("사건 수:", len(starts), "| 시작(초):", (starts / fs).tolist(), "| 끝(초):", (ends / fs).tolist())

In [ ]:
starts = np.flatnonzero(edges == 1)
ends = np.flatnonzero(edges == -1)

**3. 라벨 해상도 검사.** 2번에 이어서, 시작·끝 위치가 1초 눈금에 딱 떨어지는지 확인하세요.

In [ ]:
print("시작 % fs:", sorted(set((starts % ____).tolist())), "| 끝 % fs:", sorted(set((ends % ____).tolist())))

In [ ]:
print("시작 % fs:", sorted(set((starts % fs).tolist())), "| 끝 % fs:", sorted(set((ends % fs).tolist())))

**4. 없는 센서 찾기.** 이 데이터는 센서가 없으면 그 열을 **0으로 채웠습니다.** 작은 표에서 어느 정강이 센서가 있는지 판정하세요.

In [ ]:
import pandas as pd
df = pd.DataFrame({"RShank_ax": [0.0, 0.0, 0.0], "RShank_ay": [0.0, 0.0, 0.0], "RShank_az": [0.0, 0.0, 0.0],
                   "LShank_ax": [8019.0, 8012.3, 8013.2], "LShank_ay": [-2540.0, -2550.2, -2568.0], "LShank_az": [-1292.0, -1293.3, -1310.2]})
for s in ["LShank", "RShank"]:
    cols = [f"{s}_ax", f"{s}_ay", f"{s}_az"]
    present = (df[cols] != ____).any(axis=None)
    print(s, "있음" if present else "없음(0으로 채워짐)")

In [ ]:
    present = (df[cols] != 0).any(axis=None)

**5. 환자별 FOG 비율.** 과제 파일 4개(환자 A 두 개, B 두 개)의 표에서 환자별 FOG 비율을 구하세요.

In [ ]:
import pandas as pd
files = pd.DataFrame({"patient": ["A", "A", "B", "B"], "dur_s": [300, 60, 200, 40], "fog_s": [90, 0, 0, 4]})
for p, g in files.groupby("patient"):
    pct = 100 * g["fog_s"].sum() / g["dur_s"].____()
    print(p, "FOG %.1f%%" % pct)
print("과제 평균으로 잘못 구하면:", files.assign(r=100 * files.fog_s / files.dur_s).groupby("patient")["r"].mean().round(1).to_dict())

In [ ]:
    pct = 100 * g["fog_s"].sum() / g["dur_s"].sum()

### 실제 데이터로 해보기 (선택)

**R1 — 한 파일 열어 보기와 열 이름 붙이기.** 열 이름은 저자 코드의 `settings.toml`(EEG 순서)과 README(나머지)를 합쳐 정했습니다. 27–31열은 환자마다 뜻이 달라서 일단 `c27`…`c31`로 두고, 7번째 IMU 열은 팔(Arm)에서만 SC이고 나머지는 무효(NC)라서 `x7`로 둡니다.

In [ ]:
import numpy as np
import pandas as pd

DATA = "data/xuanwu/Filtered Data"          # Mendeley v3 zip을 data/xuanwu/에 푼 위치
fs = 500
EEG = ["FP1","FP2","F3","F4","C3","C4","P3","P4","O1","O2","F7","F8","P7","P8",
       "Fz","Cz","Pz","FC1","FC2","CP1","CP2","FC5","FC6","CP5","CP6"]
IMU = [f"{s}_{q}" for s in ["LShank", "RShank", "Waist", "Arm"]
       for q in ["ax", "ay", "az", "gx", "gy", "gz", "x7"]]          # Arm_x7 = SC, 나머지 x7 = NC
COLS = ["idx", "clock"] + EEG + ["c27", "c28", "c29", "c30", "c31"] + IMU + ["label"]

raw = pd.read_csv(f"{DATA}/001/task_1.txt", header=None)
print("모양:", raw.shape, "← README는 60열이라고 적음")
d = raw.set_axis(COLS, axis=1)
print("0번 열이 0,1,2,...인가:", bool((d["idx"].to_numpy() == np.arange(len(d))).all()))
print("1번 열 처음 3개:", d["clock"].head(3).tolist())
clk = pd.to_datetime(d["clock"], format="mixed")
print("시계 간격(ms) 종류:", np.unique(np.diff(clk.to_numpy()).astype("timedelta64[ms]").astype(int)))
print("길이: %d행 = %.3f초" % (len(d), len(d) / fs))
print("라벨 값:", sorted(d["label"].unique().tolist()))
for s in ["LShank", "RShank", "Waist", "Arm"]:
    a = d[[f"{s}_ax", f"{s}_ay", f"{s}_az"]]
    print(f"{s:7s} 0이 아닌 값 있음: {bool((a != 0).any(axis=None))!s:5s} 축 평균:", a.mean().round(0).tolist())
print("LShank_x7(NC) 고유값:", d["LShank_x7"].round(3).unique()[:5], "| Arm_x7(SC) 범위:", d["Arm_x7"].min(), "~", d["Arm_x7"].max())
print("c27–c31 표준편차:", d[["c27", "c28", "c29", "c30", "c31"]].std().round(0).tolist())

**R2 — 56개 과제 파일 전체를 세기.** 라벨과 가속도 12열만 읽어 시간을 줄입니다(전체 약 3.6 GB, 몇 분 걸림).

In [ ]:
import glob, os
import numpy as np
import pandas as pd

DATA = "data/xuanwu/Filtered Data"
fs = 500
ACC = {"LShank": [32, 33, 34], "RShank": [39, 40, 41], "Waist": [46, 47, 48], "Arm": [53, 54, 55]}  # 0부터 센 열 번호
LABEL = 60

rows, all_len = [], []
for path in sorted(glob.glob(f"{DATA}/**/task_*.txt", recursive=True)):
    rel = os.path.relpath(path, DATA)                      # 예: 001/task_1.txt, 008/OFF_1/task_1.txt
    parts = rel.split(os.sep)
    use = [LABEL] + sum(ACC.values(), [])
    d = pd.read_csv(path, header=None, usecols=use)        # 필요한 열만 읽어 메모리를 아낀다
    y = d[LABEL].to_numpy()
    edges = np.diff(np.r_[0, y, 0])
    lens = np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)   # 사건마다 행 수
    all_len += (lens / fs).tolist()
    row = {"patient": parts[0], "session": parts[1] if len(parts) == 3 else "-",
           "task": parts[-1][:-4], "dur_s": len(y) / fs, "events": len(lens),
           "fog_s": y.sum() / fs, "len_mod_500": sorted(set((lens % fs).tolist()))}
    for s, c in ACC.items():
        row[s] = bool((d[c] != 0).any(axis=None))
    rows.append(row)
files = pd.DataFrame(rows)
print("과제 파일 %d개, 환자 %d명, 실험(세션) %d개" % (
    len(files), files["patient"].nunique(), len(files[["patient", "session"]].drop_duplicates())))
print("과제 수/실험:", files.groupby(["patient", "session"]).size().value_counts().sort_index().to_dict())
print("FOG 사건 길이를 500행으로 나눈 나머지(모든 파일):",
      sorted(set(sum(files["len_mod_500"].tolist(), []))))

pt = files.groupby("patient").agg(tasks=("task", "size"), minutes=("dur_s", lambda x: x.sum() / 60),
                                  events=("events", "sum"), fog_s=("fog_s", "sum"),
                                  LShank=("LShank", "max"), RShank=("RShank", "max"),
                                  Waist=("Waist", "max"), Arm=("Arm", "max"))
pt["fog_pct"] = 100 * pt["fog_s"] / (60 * pt["minutes"])
print(pt.round(1).to_string())
print("합계: %.1f분, FOG 사건(연속 구간) %d개, FOG %.1f분" % (
    files["dur_s"].sum() / 60, files["events"].sum(), files["fog_s"].sum() / 60))
L = np.array(all_len)
print("사건 길이: %.0f–%.0f초, 5초 미만 %.0f%%, 10초 미만 %.0f%%" % (L.min(), L.max(), 100 * (L < 5).mean(), 100 * (L < 10).mean()))
print("센서가 과제마다 바뀐 환자:", [p for p, g in files.groupby("patient")
                                     if g[list(ACC)].nunique().max() > 1])
files.to_csv("K1_files.csv", index=False)

**R3 — FOG 시작 전후 30초 그림.** 001의 왼쪽 정강이 가속도 크기, 왼쪽 전경골근(L-TA) EMG, 라벨을 한 그림에 그립니다. 어느 열이 L-TA인지는 README의 "Order of 27-31 columns" 표에서 옮겼습니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA = "data/xuanwu/Filtered Data"
fs = 500
# README "Order of 27-31 columns" 표에서 왼쪽 전경골근(L-TA)이 들어 있는 열(0부터 센 번호)
L_TA = {"001": 28, "002": 28, "003": 27, "004": 27, "005": 27, "006": 28, "007": 28,
        "008/OFF_1": 28, "008/OFF_2": 31, "009": 27, "010": 27, "011": 27, "012": 27}
rec = "001"
d = pd.read_csv(f"{DATA}/{rec}/task_1.txt", header=None, usecols=[0, 32, 33, 34, L_TA[rec], 60])
d.columns = ["idx", "ax", "ay", "az", "emg", "label"]
t = d["idx"].to_numpy() / fs                            # 인덱스 → 초
y = d["label"].to_numpy()
edges = np.diff(np.r_[0, y, 0])
starts, ends = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
ok = (starts >= 30 * fs) & (ends - starts >= 10 * fs)  # 앞에 30초가 온전히 있고 10초 이상 이어진 첫 FOG
on, off = starts[ok][0], ends[ok][0]
print("FOG 시작 %.1f초, 끝 %.1f초, 길이 %.0f초" % (t[on], t[off], (off - on) / fs))
seg = slice(on - 30 * fs, on + 30 * fs)
tt = t[seg] - t[on]
acc_mag = np.linalg.norm(d[["ax", "ay", "az"]].to_numpy()[seg], axis=1)

fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [2, 2, 1]})
ax[0].plot(tt, acc_mag, lw=0.5); ax[0].set_ylabel("L shank |ACC|\n(ADC counts)")
ax[1].plot(tt, d["emg"].to_numpy()[seg], lw=0.3, color="tab:green"); ax[1].set_ylabel("L-TA EMG\n(file units)")
ax[2].step(tt, y[seg], where="post", color="tab:red"); ax[2].set_ylabel("label"); ax[2].set_yticks([0, 1])
ax[2].set_xlabel("time from FOG onset (s)")
for a in ax:
    a.axvline(0, color="k", ls="--", lw=0.8)
fig.suptitle(f"Zhang 2022 filtered data, patient {rec} task_1: ±30 s around a FOG onset")
fig.tight_layout()
os.makedirs("figures/advanced/K", exist_ok=True)
fig.savefig("figures/advanced/K/K1_real_onset.png", dpi=120)
pre, post = slice(on - 10 * fs, on), slice(on, on + 10 * fs)
e = d["emg"].to_numpy()
print("|ACC| 표준편차  시작 전 10초 %.0f, 시작 후 10초 %.0f" % (acc_mag[20*fs:30*fs].std(), acc_mag[30*fs:40*fs].std()))
print("|EMG| 평균      시작 전 10초 %.0f, 시작 후 10초 %.0f" % (np.abs(e[pre]).mean(), np.abs(e[post]).mean()))
print("저장: figures/advanced/K/K1_real_onset.png")